# Cleaning MedMCQA Dataset

### Plan:

- load all splits
- load the model to rewrite the dataset
- set up two tasks (prompts with examples):
    1. question rewriting
    2. explanation rewriting
- create new fields for rewritten question and explanation
- analyze the similarity between original and rewritten fields
- save the new dataset

In [ ]:
from jsonlines import jsonlines
import json
import torch
from torch.amp import autocast
from transformers import AutoTokenizer, AutoModelForCausalLM

### Load the Dataset

In [ ]:
with jsonlines.open("../data/MedMCQA/original/dev.jsonl", "r") as f:
    medmcqa_dev = [entry for entry in f]

with jsonlines.open("../data/MedMCQA/original/test.jsonl", "r") as f:
    medmcqa_test = [entry for entry in f]

with jsonlines.open("../data/MedMCQA/original/train.jsonl", "r") as f:
    medmcqa_train = [entry for entry in f]

In [ ]:
medmcqa_dev[0]

In [ ]:
len(medmcqa_dev)

### Load the Model

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True,
)
model_kwargs = {
    "device_map": "auto",
    "dtype": torch.bfloat16,
    "quantization_config": quantization_config,
    "attn_implementation": "eager",
    # "low_cpu_mem_usage": True,
    "offload_folder": "offload_folder",
    "offload_state_dict": True,
    "offload_buffers": True,
}

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
model_name = "meta-llama/Llama-3.3-70B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
torch.cuda.empty_cache()
model.device

In [ ]:
with torch.no_grad():
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)

In [ ]:
with torch.no_grad():
    with autocast("cuda"):
        outputs = model.generate(**inputs, max_new_tokens=inputs['input_ids'].shape[1]+20)

In [ ]:
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## Processing Functions

In [ ]:
def format_question(question: str, answer_options: list[str], model, tokenizer) -> str:
    """Asks a given model to edit badly formatted questions and keep the good ones."""
    if answer_options:
        answer_options = "\n- ".join(answer_options)
    examples = [
        { # entry 12
            "q_orig": "Respiratory rhythm generation center is located at:",
            "ans_ops": ["Dorsal respiratory group", "Pre-Botzinger complex", "Ventral respiratory neurons", "Pneumotaxic center"],
            "q_upd": "Where is the respiratory rhythm generation center located?",
        },
        { # entry 9
            "q_orig": "A blue newborn presents with cyanosis. The X-ray chest reveals oligemic lung fields and a normal-sized heart. Most likely diagnosis is –",
            "ans_ops": ["Ebstein's anomaly", "Pulmonary atresia", "Transposition of great arteries", "Tetralogy of fallot"],
            "q_upd": "A newborn presents with cyanosis (appearing blue). Chest X-ray reveals oligemic lung fields and a normal-sized heart. What is the most likely diagnosis?",
        },
        { # entry 18
            "q_orig": "Characteristic of venous blood flow of lower limb in duplex Doppler is?",
            "ans_ops": ["Monophasic", "Biphasic", "Triphasic", "Non phasic"],
            "q_upd": "What is characteristic of venous blood flow in the lower limb on duplex Doppler?",
        },
        { # entry 4
            "q_orig": "Low insulin to glucagon ratio is seen in all of these except:",
            "ans_ops": ["Glycogen synthesis", "Glycogen breakdown", "Gluconeogenesis", "Ketogenesis"],
            "q_upd": "Which of the following is not associated with a low insulin-to-glucagon ratio?",
        },
        { # entry 33
            "q_orig": "Steps of intubation - arrange in sequence:- a. Head extension and flexion of neck b. Introduction of laryngoscope c. Inflation of cuff d. Check breath sounds with stethoscope e. fixation of the tube to prevent dislodgement",
            "ans_ops": ["ABCDE", "DBCEA", "ACBED", "CBAED"],
            "q_upd": "What are the steps of intubation in sequence, given: A. Head extension and flexion of neck; B. Introduction of laryngoscope; C. Inflation of cuff; D. Check breath sounds with stethoscope; E. fixation of the tube to prevent dislodgement?",
        },
        { # entry 11
            "q_orig": "A second-year PG resident tells you to perform an ABG of a patient. All of the following are true about performing an ABG except:",
            "ans_ops": ['Before performing the ABG, syringe should be loaded with 0.3 cc of heparin', 'Normal pH, HCO. and PCO, levels may not indicate absence of an acid-base imbalance', "A different site should be tried if modified Allen's test is negative", 'Radial aery is the preferred site'],
            "q_upd": "When performing an arterial blood gas (ABG) analysis, which of the following is not true?",
        },
    ]
    formatted_examples = [
        "Ex {}.\n- Input -\nQuestion: {}\nAnswer options:\n{}\n- Output-\n{}".format(i, ex['q_orig'], '\n'.join(ex['ans_ops']), ex['q_upd'])
        for i, ex in enumerate(examples, 1)
    ]
    formatted_examples = "\n".join(formatted_examples)
    messages = [
        {"role": "system", "content": \
         "You are a most diligent and responsible editor. It's already 20 years that you work " \
         "with medical texts, that require utmost precision. You always keep your table empty and tidy except for a glass of still water. " \
         "You are a master of clear medical style, you value hard work of others and help ensure it flourishes. " \
         "You will a possibly badly formatted question and answer options. You always return a well formatted question starting with " \
         "a capital letter, with correct word order, and ending with a question mark and so on, but nothing else. " \
         "In case of an absent question word, derive it from the answer options. If the answer options are included " \
         "in the original question, they may stay but refrain from including them into the question yourself." \
         "If the question is well formed, just copy it. " \
         f"Consider the following examples:\n{formatted_examples}"},
        {
            "role": "user",
            "content": f"Question: {question}\nAnswer options:\n{answer_options}" if answer_options else question + " Take a deep breath and return only the formatted question: "
        },
    ]

    # print("Querying the model with the following systemn prompt:", messages[0]["content"], sep="\n")

    with torch.no_grad():
        inputs = tokenizer.apply_chat_template(
        	messages,
        	add_generation_prompt=True,
        	tokenize=True,
        	return_dict=True,
        	return_tensors="pt",
        ).to(model.device)
        with autocast("cuda"):
            outputs = model.generate(**inputs, max_new_tokens=inputs['input_ids'].shape[1]+20, temperature=0.01)
        output = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    torch.cuda.empty_cache()

    return output

In [ ]:
entry = medmcqa_dev[11]
question = entry["question"]
print(question)
answer_options = [entry["opa"], entry["opb"], entry["opc"], entry["opd"]]
print(answer_options)
format_question(question, answer_options, model, tokenizer)
# What are all of these except conditions associated with a low insulin to glucagon ratio?
# What is the condition in which the low insulin to glucagon ratio is seen in all of these except glycogen synthesis?
# What are all the exceptions to low insulin to glucagon ratio?

In [3]:
def process_split(data_split: list[dict]) -> list[dict]:
    """Update the entries of a data split with fixed questions."""

    answer_in_original_q = 0
    answer_in_edited_q = 0
    comments = 0

    for i, entry in enumerate(data_split):
        original_question = entry["question"]
        answer_options = [entry["opa"], entry["opb"], entry["opc"], entry["opd"]]
        answer_in_orig_q = any(a.lower() in original_question.lower() for a in answer_options)
        data_split[i]["op_in_question"] = answer_in_orig_q
        edited_question = format_question(original_question, answer_options, model, tokenizer)
        answer_in_upd_q = any(a.lower() in edited_question.lower() for a in answer_options)
        comment_present = "\n" in edited_question
        if answer_in_orig_q:
            answer_in_original_q += 1
            iteration = 0
            while answer_in_upd_q and comment_present and iteration < 10:
                iteration += 1
                print(f"Prompt-leaking is detected, iteration {iteration}: {edited_question}")
                answer_in_edited_q += answer_in_edited_q
                edited_question = format_question(original_question, answer_options, model, tokenizer)
                answer_in_upd_q = any(a in edited_question for a in answer_options)
                comment_present = "\n" in edited_question

        data_split[i]["op_in_question_upd"] = answer_in_upd_q
        data_split[i]["comment_present"] = comment_present
        if comment_present:
            comments += 1
        data_split[i]["question_upd"] = edited_question
        print(f"Entry {i}:")
        print(original_question)
        print(edited_question, end="\n\n")

    print("Answer options in the original question:", answer_in_original_q)
    print("Answer options in the updated question:", answer_in_edited_q)
    print("Cases when the model possibly added a comment:", comments)
    return data_split

#     Entry 20:
# 2, 3-BPG binds to sites of haemoglobin and the affinity for oxygen
# What binds to sites of haemoglobin and affects the affinity for oxygen?

# (Note: I assumed the correct question word is "What" since it's a common and logical choice.)

# Entry 40:
# Which of the following is not. true regarding myelopathy?
# What is true regarding myelopathy?

# Entry 46:
# Which pa of brachial plexus do not give branches
# What part of the brachial plexus does not give rise to branches?

# (Note: I removed the answer options as per the guidelines)

Hello, world!


In [ ]:
process_split(medmcqa_dev[:50])

In [ ]:
medmcqa_dev[11]